# OpenAI File-Based ASR: Transcription, Speaker Diarization, and Streaming

This notebook implements the complete **file-audio processing stage** for a speech analytics pipeline using the OpenAI Audio Transcriptions API.

## Covered topics

1. Architecture and preprocessing decisions
2. Environment setup
3. Audio-file validation
4. Standard ASR with `gpt-4o-transcribe`
5. Explanation of transcription parameters
6. Reusable transcription function
7. Speaker diarization with `gpt-4o-transcribe-diarize`
8. Automatic and custom server-side VAD chunking
9. Known-speaker references
10. File-based streaming without diarization
11. File-based streaming with diarization
12. Saving text, JSON, JSONL, and segment outputs
13. Error handling and production recommendations

> **Important:** File-based streaming means the completed audio file is uploaded and transcript events are returned progressively while it is processed. It is not continuous microphone or telephone audio streaming. For ongoing audio, use OpenAI Realtime transcription.


## 1. Where this notebook fits in the architecture

```text
CUSTOMER CALL / MEETING
          │
          ▼
AUDIO INGESTION
Telephony │ Meeting platform │ Uploaded recording
          │
          ▼
AUDIO VALIDATION AND CONDITIONAL PREPROCESSING
Format │ File size │ Corruption │ Optional cleanup
          │
          ▼
OPENAI FILE TRANSCRIPTION API
Standard ASR or ASR + diarization
          │
          ▼
TRANSCRIPT / SPEAKER SEGMENTS
          │
          ▼
Cleaning │ Redaction │ Analytics │ LLM │ Storage
```

OpenAI can process common audio files directly. Preprocessing should therefore be **conditional**, not automatically applied to every file.

Recommended decisions:

| Step | Recommendation |
|---|---|
| File existence and size validation | Always |
| Format validation | Always |
| Resampling | Only for unsupported or raw telephony audio |
| Mono conversion | Usually unnecessary; preserve stereo when channels represent speakers |
| Noise reduction | Only for poor-quality audio |
| Volume normalization | Only for unusually low or uneven audio |
| VAD/chunking | Useful for long files, silence, hold time, retries, or diarization |
| Channel splitting | Prefer it when agent and customer are recorded on separate channels |


## 2. API constraints used in this notebook

At the time this notebook was prepared:

- File transcription accepts uploaded, completed recordings.
- Individual file uploads can be up to **25 MB**.
- Common supported formats include `flac`, `mp3`, `mp4`, `mpeg`, `mpga`, `m4a`, `ogg`, `wav`, and `webm`.
- `gpt-4o-transcribe` supports normal transcription and file streaming.
- `gpt-4o-transcribe-diarize` provides speaker-labelled segments.
- For diarized audio longer than 30 seconds, `chunking_strategy` must be set to `"auto"` or a server-VAD configuration.
- The diarization model does not support `prompt`, `logprobs`, or custom timestamp granularities.
- File streaming emits:
  - `transcript.text.delta`
  - `transcript.text.segment` for finalized diarized segments
  - `transcript.text.done`

Always check the current OpenAI API documentation before production deployment because model capabilities can change.


## 3. Install dependencies

In [ ]:
%pip install --upgrade openai python-dotenv

## 4. Configure the API key

Create a `.env` file in the notebook directory:

```env
OPENAI_API_KEY=your_openai_api_key_here
```

Do not commit the `.env` file to source control.


In [ ]:
import os
import json
import base64
import mimetypes
from pathlib import Path
from typing import Any, Optional

from dotenv import load_dotenv
from openai import OpenAI
from openai import (
    APIConnectionError,
    APIStatusError,
    AuthenticationError,
    BadRequestError,
    RateLimitError,
)

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. Add it to a .env file "
        "or set it as an environment variable."
    )

client = OpenAI(
    api_key=api_key,
    timeout=600.0,
    max_retries=3,
)

print("OpenAI client initialized.")


## 5. Configure input and output paths

In [ ]:
AUDIO_PATH = Path("customer_call.mp3")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_FORMATS = {
    ".flac",
    ".mp3",
    ".mp4",
    ".mpeg",
    ".mpga",
    ".m4a",
    ".ogg",
    ".wav",
    ".webm",
}

MAX_UPLOAD_BYTES = 25 * 1024 * 1024

print("Input :", AUDIO_PATH)
print("Output:", OUTPUT_DIR.resolve())


## 6. Validate the audio file

In [ ]:
def validate_audio_file(audio_path: Path) -> None:
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Audio file not found: {audio_path.resolve()}"
        )

    if not audio_path.is_file():
        raise ValueError(f"Path is not a file: {audio_path}")

    if audio_path.suffix.lower() not in SUPPORTED_FORMATS:
        raise ValueError(
            f"Unsupported format: {audio_path.suffix}. "
            f"Supported formats: {sorted(SUPPORTED_FORMATS)}"
        )

    file_size = audio_path.stat().st_size

    if file_size == 0:
        raise ValueError("The audio file is empty.")

    if file_size > MAX_UPLOAD_BYTES:
        raise ValueError(
            f"File is {file_size / (1024 ** 2):.2f} MB, which is over "
            "the 25 MB per-file upload limit. Split or compress it first."
        )

    print(f"Valid audio file: {audio_path}")
    print(f"Size: {file_size / (1024 ** 2):.2f} MB")


validate_audio_file(AUDIO_PATH)


# Part A — Standard file transcription

Use `gpt-4o-transcribe` when you need accurate speech-to-text but do not require speaker labels.

This request can include:

- `file`
- `model`
- `language`
- `prompt`
- `response_format`
- `temperature`
- `include=["logprobs"]`
- `stream`


## 7. Standard transcription with the main parameters

In [ ]:
try:
    with AUDIO_PATH.open("rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            file=audio_file,
            model="gpt-4o-transcribe",

            # Optional ISO-639-1 language hint.
            # Remove this field when the language is unknown.
            language="en",

            # Context for names, terminology, abbreviations, and style.
            prompt=(
                "This is a customer-support call between an agent and "
                "a customer. Correctly transcribe product names, technical "
                "terms, numbers, amounts, KYC, EMI, CRM, and API. "
                "Use proper punctuation."
            ),

            # gpt-4o-transcribe supports JSON output.
            response_format="json",

            # Low temperature is preferred for deterministic ASR.
            temperature=0.0,

            # Available for selected non-diarization transcription models.
            include=["logprobs"],

            # Complete response instead of incremental events.
            stream=False,
        )

    print(transcription.text)

except AuthenticationError:
    print("Authentication failed. Check OPENAI_API_KEY.")
    raise
except BadRequestError as error:
    print(f"Invalid transcription request: {error}")
    raise
except RateLimitError:
    print("Rate limit or API quota exceeded.")
    raise
except APIConnectionError:
    print("Could not connect to the OpenAI API.")
    raise
except APIStatusError as error:
    print(f"OpenAI API error {error.status_code}: {error}")
    raise


## 8. Standard transcription parameter reference

### `file`
The binary file object opened with `rb`.

### `model`
`gpt-4o-transcribe` performs ordinary speech-to-text. A newer general-purpose option may also be available as `gpt-transcribe`.

### `language`
An optional ISO-639-1 code such as:

```python
language="en"  # English
language="hi"  # Hindi
```

Omit it for automatic detection or highly mixed-language audio.

### `prompt`
Use it to improve spellings, acronyms, names, domain vocabulary, punctuation, or continuity from a previous chunk. It is not a summarization prompt.

### `response_format`
For `gpt-4o-transcribe`, use:

```python
response_format="json"
```

### `temperature`
A number from 0 to 1. Low values are appropriate for consistent transcription.

### `include`
`include=["logprobs"]` returns token log probabilities for supported transcription models when JSON output is used. It is not supported by the diarization model.

### `stream`
- `False`: wait for the complete response
- `True`: receive transcript events progressively


## 9. Save standard transcription outputs

In [ ]:
standard_result = transcription.model_dump()

standard_json_path = OUTPUT_DIR / f"{AUDIO_PATH.stem}_transcription.json"
standard_text_path = OUTPUT_DIR / f"{AUDIO_PATH.stem}_transcription.txt"

standard_json_path.write_text(
    json.dumps(standard_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

standard_text_path.write_text(
    transcription.text,
    encoding="utf-8",
)

print("JSON:", standard_json_path.resolve())
print("Text:", standard_text_path.resolve())


## 10. Inspect usage and token confidence

In [ ]:
if getattr(transcription, "usage", None):
    usage = (
        transcription.usage.model_dump()
        if hasattr(transcription.usage, "model_dump")
        else transcription.usage
    )
    print("Usage:")
    print(json.dumps(usage, indent=2))

if getattr(transcription, "logprobs", None):
    print("\nFirst 10 token log probabilities:")
    for item in transcription.logprobs[:10]:
        data = item.model_dump() if hasattr(item, "model_dump") else item
        print(data)


## 11. Reusable standard transcription function

In [ ]:
def transcribe_audio(
    audio_path: str | Path,
    language: Optional[str] = None,
    domain_prompt: Optional[str] = None,
    output_directory: str | Path = "outputs",
    model: str = "gpt-4o-transcribe",
) -> dict[str, Any]:
    audio_path = Path(audio_path)
    output_directory = Path(output_directory)

    validate_audio_file(audio_path)
    output_directory.mkdir(parents=True, exist_ok=True)

    request_parameters: dict[str, Any] = {
        "model": model,
        "response_format": "json",
        "temperature": 0.0,
        "include": ["logprobs"],
        "stream": False,
    }

    if language:
        request_parameters["language"] = language

    if domain_prompt:
        request_parameters["prompt"] = domain_prompt

    with audio_path.open("rb") as audio_file:
        response = client.audio.transcriptions.create(
            file=audio_file,
            **request_parameters,
        )

    result = response.model_dump()

    (output_directory / f"{audio_path.stem}_transcription.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    (output_directory / f"{audio_path.stem}_transcription.txt").write_text(
        response.text,
        encoding="utf-8",
    )

    return result


In [ ]:
# Example usage:
#
# result = transcribe_audio(
#     audio_path="customer_call.mp3",
#     language="en",
#     domain_prompt=(
#         "Customer-service conversation. Common terms include "
#         "OpenAI, KYC, EMI, CRM, API, and agent assist."
#     ),
# )
#
# print(result["text"])


# Part B — Speaker diarization

Use `gpt-4o-transcribe-diarize` when you need:

```text
What was said
+
Who spoke when
```

The output can contain segments such as:

```text
[0.00s - 4.30s] Speaker A: Hello, how can I help?
[4.31s - 8.90s] Speaker B: I need help with my account.
```


## 12. Basic diarized transcription

In [ ]:
try:
    with AUDIO_PATH.open("rb") as audio_file:
        diarized = client.audio.transcriptions.create(
            file=audio_file,
            model="gpt-4o-transcribe-diarize",

            # Required to receive speaker annotations.
            response_format="diarized_json",

            # Required for diarized files longer than 30 seconds.
            chunking_strategy="auto",

            # Optional language hint.
            language="en",

            # Low value for consistent output.
            temperature=0.0,

            # Complete response.
            stream=False,
        )

    print("Complete transcript:\n")
    print(diarized.text)

    print("\nSpeaker segments:\n")
    for segment in diarized.segments:
        print(
            f"[{segment.start:.2f}s - {segment.end:.2f}s] "
            f"Speaker {segment.speaker}: {segment.text}"
        )

except AuthenticationError:
    print("Authentication failed. Check OPENAI_API_KEY.")
    raise
except BadRequestError as error:
    print(f"Invalid diarization request: {error}")
    raise
except RateLimitError:
    print("Rate limit or API quota exceeded.")
    raise
except APIConnectionError:
    print("Could not connect to the OpenAI API.")
    raise
except APIStatusError as error:
    print(f"OpenAI API error {error.status_code}: {error}")
    raise


## 13. Diarization parameter reference

### Supported/recommended parameters

```python
file=audio_file
model="gpt-4o-transcribe-diarize"
response_format="diarized_json"
chunking_strategy="auto"
language="en"
temperature=0.0
stream=False
```

### `response_format="diarized_json"`
Required for speaker annotations. Each finalized segment contains:

- `id`
- `start`
- `end`
- `speaker`
- `text`
- `type`

Unknown speakers are normally labelled `A`, `B`, `C`, and so on.

### `chunking_strategy="auto"`
The server normalizes loudness and uses server-side VAD to select chunk boundaries. It is required when a diarized input is longer than 30 seconds.

### Unsupported with the diarization model

Do not pass:

```python
prompt="..."
include=["logprobs"]
timestamp_granularities=["word"]
```

The diarized response already provides segment timestamps, not word-level timestamp configuration.


## 14. Save diarized outputs

In [ ]:
diarized_result = diarized.model_dump()

diarized_json_path = OUTPUT_DIR / f"{AUDIO_PATH.stem}_diarized.json"
diarized_text_path = OUTPUT_DIR / f"{AUDIO_PATH.stem}_diarized.txt"

diarized_json_path.write_text(
    json.dumps(diarized_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

lines = [
    (
        f"[{segment.start:.2f}s - {segment.end:.2f}s] "
        f"Speaker {segment.speaker}: {segment.text.strip()}"
    )
    for segment in diarized.segments
]

diarized_text_path.write_text(
    "\n\n".join(lines),
    encoding="utf-8",
)

print("JSON:", diarized_json_path.resolve())
print("Text:", diarized_text_path.resolve())


## 15. Custom server-side VAD chunking

In [ ]:
custom_vad = {
    "type": "server_vad",

    # Audio retained before detected speech.
    "prefix_padding_ms": 400,

    # Silence needed to mark speech end.
    "silence_duration_ms": 600,

    # 0.0–1.0. Higher requires louder/clearer speech.
    "threshold": 0.4,
}

# Example:
#
# with AUDIO_PATH.open("rb") as audio_file:
#     diarized_custom_vad = client.audio.transcriptions.create(
#         file=audio_file,
#         model="gpt-4o-transcribe-diarize",
#         response_format="diarized_json",
#         chunking_strategy=custom_vad,
#         language="en",
#         temperature=0.0,
#         stream=False,
#     )


### VAD tuning guidance

| Parameter | Effect |
|---|---|
| `prefix_padding_ms` | Preserves audio immediately before detected speech |
| `silence_duration_ms` | Shorter values create faster, smaller turns; longer values avoid splitting natural pauses |
| `threshold` | Higher values reject more low-volume audio/noise; lower values capture quieter speech |

Start with `"auto"` unless you have evaluated audio-specific VAD requirements.


## 16. Reusable diarization function

In [ ]:
def diarize_audio(
    audio_path: str | Path,
    language: Optional[str] = None,
    output_directory: str | Path = "outputs",
    chunking_strategy: str | dict[str, Any] = "auto",
) -> dict[str, Any]:
    audio_path = Path(audio_path)
    output_directory = Path(output_directory)

    validate_audio_file(audio_path)
    output_directory.mkdir(parents=True, exist_ok=True)

    parameters: dict[str, Any] = {
        "model": "gpt-4o-transcribe-diarize",
        "response_format": "diarized_json",
        "chunking_strategy": chunking_strategy,
        "temperature": 0.0,
        "stream": False,
    }

    if language:
        parameters["language"] = language

    with audio_path.open("rb") as audio_file:
        response = client.audio.transcriptions.create(
            file=audio_file,
            **parameters,
        )

    result = response.model_dump()

    json_path = output_directory / f"{audio_path.stem}_diarized.json"
    text_path = output_directory / f"{audio_path.stem}_diarized.txt"

    json_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    readable_lines = [
        (
            f"[{segment.start:.2f}s - {segment.end:.2f}s] "
            f"Speaker {segment.speaker}: {segment.text.strip()}"
        )
        for segment in response.segments
    ]

    text_path.write_text(
        "\n\n".join(readable_lines),
        encoding="utf-8",
    )

    return result


# Part C — Known-speaker references

Normally, the diarization model returns generic labels such as `A` and `B`.

You may provide up to four known speakers using:

- `known_speaker_names`
- `known_speaker_references`

Each reference should be a clean 2–10 second clip containing one speaker. The order of names and references must match.


In [ ]:
def audio_to_data_url(audio_path: str | Path) -> str:
    path = Path(audio_path)

    if not path.exists():
        raise FileNotFoundError(path)

    mime_type, _ = mimetypes.guess_type(path)

    if mime_type is None:
        mime_type = "audio/wav"

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"


In [ ]:
# Uncomment after preparing clean 2–10 second reference clips.
#
# agent_reference = audio_to_data_url("agent_sample.wav")
# customer_reference = audio_to_data_url("customer_sample.wav")
#
# with AUDIO_PATH.open("rb") as audio_file:
#     identified = client.audio.transcriptions.create(
#         file=audio_file,
#         model="gpt-4o-transcribe-diarize",
#         response_format="diarized_json",
#         chunking_strategy="auto",
#         language="en",
#         temperature=0.0,
#         stream=False,
#         known_speaker_names=["agent", "customer"],
#         known_speaker_references=[
#             agent_reference,
#             customer_reference,
#         ],
#     )
#
# for segment in identified.segments:
#     print(
#         f"[{segment.start:.2f}s - {segment.end:.2f}s] "
#         f"{segment.speaker}: {segment.text}"
#     )


# Part D — File-based streaming

File-based streaming:

```text
Completed audio file
        │
        ▼
Upload through /audio/transcriptions
        │
        ▼
Receive partial transcript events
```

It does not send microphone chunks continuously. For live calls or microphone audio, use the Realtime transcription API.


## 17. Minimal standard file-streaming example

In [ ]:
# This streams text for a completed file without speaker diarization.
#
# with AUDIO_PATH.open("rb") as audio_file:
#     stream = client.audio.transcriptions.create(
#         file=audio_file,
#         model="gpt-4o-transcribe",
#         response_format="json",
#         language="en",
#         temperature=0.0,
#         stream=True,
#     )
#
#     for event in stream:
#         if event.type == "transcript.text.delta":
#             print(event.delta, end="", flush=True)
#
#         elif event.type == "transcript.text.done":
#             print("\n\nTranscription completed.")
#             print(event.text)


## 18. Streaming event types

### `transcript.text.delta`
Contains partial text. With diarization, it can include a `segment_id`, but the speaker is not finalized yet.

### `transcript.text.segment`
Emitted by the diarization model when a segment is finalized. It includes final speaker, timestamps, and text.

### `transcript.text.done`
Indicates completion and contains the complete transcript.


## 19. Complete diarized file-streaming implementation

In [ ]:
def event_to_dict(event: Any) -> dict[str, Any]:
    if hasattr(event, "model_dump"):
        return event.model_dump()

    if isinstance(event, dict):
        return event

    return {
        "type": getattr(event, "type", None),
        "delta": getattr(event, "delta", None),
        "id": getattr(event, "id", None),
        "segment_id": getattr(event, "segment_id", None),
        "speaker": getattr(event, "speaker", None),
        "start": getattr(event, "start", None),
        "end": getattr(event, "end", None),
        "text": getattr(event, "text", None),
    }


In [ ]:
def stream_diarized_file(
    audio_path: str | Path,
    language: Optional[str] = None,
    output_directory: str | Path = "outputs",
    chunking_strategy: str | dict[str, Any] = "auto",
) -> dict[str, Any]:
    audio_path = Path(audio_path)
    output_directory = Path(output_directory)

    validate_audio_file(audio_path)
    output_directory.mkdir(parents=True, exist_ok=True)

    jsonl_path = output_directory / f"{audio_path.stem}_stream_events.jsonl"
    text_path = output_directory / f"{audio_path.stem}_streamed_diarized.txt"
    json_path = output_directory / f"{audio_path.stem}_streamed_segments.json"

    request_parameters: dict[str, Any] = {
        "model": "gpt-4o-transcribe-diarize",
        "response_format": "diarized_json",
        "chunking_strategy": chunking_strategy,
        "temperature": 0.0,
        "stream": True,
    }

    if language:
        request_parameters["language"] = language

    all_segments: list[dict[str, Any]] = []
    complete_text = ""

    with audio_path.open("rb") as audio_file:
        stream = client.audio.transcriptions.create(
            file=audio_file,
            **request_parameters,
        )

        print("Streaming transcription started.\n")

        with jsonl_path.open("w", encoding="utf-8") as jsonl_file:
            for event in stream:
                event_data = event_to_dict(event)
                event_type = event_data.get("type")

                jsonl_file.write(
                    json.dumps(event_data, ensure_ascii=False) + "\n"
                )

                if event_type == "transcript.text.delta":
                    delta = event_data.get("delta") or ""
                    print(delta, end="", flush=True)

                elif event_type == "transcript.text.segment":
                    segment = {
                        "id": event_data.get("id"),
                        "speaker": event_data.get("speaker"),
                        "start": event_data.get("start"),
                        "end": event_data.get("end"),
                        "text": (event_data.get("text") or "").strip(),
                    }
                    all_segments.append(segment)

                    start = segment["start"]
                    end = segment["end"]

                    start_text = f"{start:.2f}" if isinstance(start, (int, float)) else "?"
                    end_text = f"{end:.2f}" if isinstance(end, (int, float)) else "?"

                    print(
                        "\n\n"
                        f"[FINAL SEGMENT] {start_text}s - {end_text}s | "
                        f"Speaker {segment['speaker']}: {segment['text']}\n"
                    )

                elif event_type == "transcript.text.done":
                    complete_text = event_data.get("text") or ""
                    print("\n\nTranscription completed.")

    readable_lines = []

    for segment in all_segments:
        start = segment["start"]
        end = segment["end"]
        start_text = f"{start:.2f}" if isinstance(start, (int, float)) else "?"
        end_text = f"{end:.2f}" if isinstance(end, (int, float)) else "?"

        readable_lines.append(
            f"[{start_text}s - {end_text}s] "
            f"Speaker {segment['speaker']}: {segment['text']}"
        )

    text_path.write_text(
        "\n\n".join(readable_lines),
        encoding="utf-8",
    )

    final_result = {
        "audio_file": str(audio_path),
        "complete_text": complete_text,
        "number_of_segments": len(all_segments),
        "segments": all_segments,
    }

    json_path.write_text(
        json.dumps(final_result, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("\nSaved files:")
    print("Raw events:", jsonl_path.resolve())
    print("Transcript:", text_path.resolve())
    print("Segments:", json_path.resolve())

    return final_result


In [ ]:
# Execute file-based streaming with diarization:
#
# streamed_result = stream_diarized_file(
#     audio_path=AUDIO_PATH,
#     language="en",
#     output_directory=OUTPUT_DIR,
#     chunking_strategy="auto",
# )


## 20. Hindi and Hindi-English calls

For a Hindi-dominant call:

```python
language="hi"
```

For mixed Hindi-English audio, compare these approaches:

1. Use `language="hi"` when Hindi is dominant.
2. Omit `language` and allow automatic detection.
3. Evaluate both approaches on a representative validation set.

Do not assume that one language setting is universally best for code-switched calls.


## 21. Production recommendations

### Use standard transcription when

- only text is needed;
- diarization is unnecessary;
- you need prompting or log probabilities;
- channels already identify speakers.

### Use diarized transcription when

- speaker labels are required;
- meetings have multiple participants;
- a mono call recording mixes agent and customer;
- you need segment start/end timestamps.

### Prefer channel splitting when

A stereo telephony file has:

```text
Left channel  → Agent
Right channel → Customer
```

In that case, splitting and transcribing each channel separately can be more reliable than predicting speaker identity.

### For files over 25 MB

- compress to an efficient supported format;
- split into logical chunks;
- preserve a small overlap;
- carry context between standard-ASR chunks using `prompt`;
- merge timestamps carefully;
- avoid independently diarizing arbitrary chunks unless you have a strategy for maintaining speaker-label consistency.

### Store both raw and normalized outputs

Keep:

1. raw API response;
2. readable transcript;
3. speaker segments;
4. processing metadata;
5. model name and parameters;
6. failure/retry information.


## 22. Final processing flow

```text
Audio file
   │
   ▼
Validate path, format, and size
   │
   ├── No speaker labels needed
   │       ▼
   │   gpt-4o-transcribe
   │       ▼
   │   Text + optional usage/logprobs
   │
   └── Speaker labels needed
           ▼
       gpt-4o-transcribe-diarize
           ▼
       Speaker segments + timestamps
           │
           ├── stream=False → final response
           └── stream=True  → delta, segment, done events

Outputs
   ├── .txt readable transcript
   ├── .json structured result
   └── .jsonl raw streaming events
```


## 23. Next stage in the complete speech-analytics system

The outputs from this notebook can feed:

- punctuation and cleaning;
- PII redaction;
- sentiment and emotion;
- intent and topic classification;
- entities and objections;
- compliance checks;
- summary and action items;
- lead scoring;
- next-best action;
- RAG over policies and product documents;
- structured database storage;
- CRM, dashboards, alerts, and agent assist.
